## Weekly shift handover (live MongoDB)

Loads shift reports from the CastNet **`reports`** collection for a date range, groups by **`equipment`** (DCM), and asks the model for a short paragraph + bullets per machine. Configure MongoDB via env vars (`CAST_LLM_MONGODB_URI`, etc.) or defaults in [settings.py](../src/cast_llm/settings.py).

In [ ]:
import json
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

from IPython.display import Markdown, display
from ollama import chat

from cast_llm import fetch_reports_date_range, get_project_root, group_by_equipment

### Date range (UK plant; Mongo `date_iso` is UTC)

Pick the wall-clock window in `Europe/London`, then convert to UTC for the query. Adjust the example datetimes to the week you need (e.g. the week before a Sunday 06:00 handover).

In [ ]:
UK = ZoneInfo("Europe/London")
UTC = timezone.utc

# Example: one calendar week in UK local time
start_local = datetime(2026, 3, 29, 0, 0, 0, tzinfo=UK)
end_local = datetime(2026, 4, 4, 23, 59, 59, tzinfo=UK)

start_utc = start_local.astimezone(UTC)
end_utc = end_local.astimezone(UTC)
print("Query UTC range:", start_utc, "→", end_utc)

### Fetch from MongoDB and group by DCM

In [ ]:
rows = fetch_reports_date_range(start_utc, end_utc)
by_dcm = group_by_equipment(rows)
print(f"Total reports: {len(rows)}  |  DCM keys: {len(by_dcm)}")
for k, v in sorted(by_dcm.items(), key=lambda kv: (kv[0] == "", kv[0])):
    label = k if k else "(no equipment)"
    print(f"  {label}: {len(v)} report(s)")

### System prompt

In [ ]:
with open(
    f"{get_project_root()}/prompts/weekly_handover_system.md", encoding="utf-8"
) as f:
    system_prompt = f.read()

### Run the model (one call per DCM)

Change `OLLAMA_MODEL` to match a model you have pulled locally.

In [ ]:
OLLAMA_MODEL = "gpt-oss:20b"

sections: list[str] = []
sections.append(
    f"# Weekly handover summary\n\n"
    f"**UTC range:** {start_utc.isoformat()} – {end_utc.isoformat()}  \n"
    f"**UK window:** {start_local.date()} – {end_local.date()}\n"
)

for equipment in sorted(by_dcm.keys(), key=lambda x: (x == "", x)):
    reports = by_dcm[equipment]
    if not reports:
        continue
    label = equipment if equipment else "(no equipment set)"
    user_content = (
        f"## Range (UTC)\n{start_utc.isoformat()} – {end_utc.isoformat()}\n"
        f"(Mongo dates are UTC; plant operates UK time.)\n\n"
        f"## equipment\n{json.dumps(label)}\n\n"
        f"## Shift reports (JSON array)\n{json.dumps(reports, indent=2)}"
    )
    stream = chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        stream=True,
        keep_alive=-1,
        options={"num_ctx": 128000},
    )
    chunks: list[str] = []
    for chunk in stream:
        chunks.append(chunk["message"]["content"])
    body = "".join(chunks)
    sections.append(f"## {label}\n\n{body}\n")
    print(f"Done: {label} ({len(reports)} reports)")

full_response = "\n---\n\n".join(sections)

### Preview

In [ ]:
display(Markdown(full_response))

### Save to `notebooks/weekly_handover.md`

In [ ]:
out_path = f"{get_project_root()}/notebooks/weekly_handover.md"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(full_response)
print("Wrote", out_path)